# Elastic Net Regression

## Objective

In this notebook, we will practice:

- Linear Regression baseline
- Elastic Net Regression
- L1 Regularization
- L2 Regularization
- Alpha parameter
- L1 Ratio
- Coefficient shrinkage
- Feature selection
- Model evaluation
- Comparing different alpha values
- Comparing different l1_ratio values

## Dataset

Heart Failure Clinical Records Dataset

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import ElasticNet

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
df = pd.read_csv(
    "../../heart_failure_clinical_records_dataset-selected-columns.csv"
)

df.head()

In [ ]:
print("Shape:", df.shape)

print(
    "Missing Values:",
    df.isnull().sum().sum()
)

df.describe()

# Preparing Data

We will use the following numerical features:

- age
- creatinine_phosphokinase
- ejection_fraction
- platelets
- serum_sodium

Target:

- serum_creatinine

In [ ]:
features = [
    "age",
    "creatinine_phosphokinase",
    "ejection_fraction",
    "platelets",
    "serum_sodium"
]

X = df[features]

y = df["serum_creatinine"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Rows:", len(X_train))
print("Testing Rows:", len(X_test))

# Elastic Net Regression

Elastic Net combines:

- L1 Regularization from Lasso
- L2 Regularization from Ridge

Important parameters:

- alpha → overall regularization strength
- l1_ratio → balance between L1 and L2

In [7]:
elastic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        ElasticNet(
            alpha=0.1,
            l1_ratio=0.5,
            max_iter=10000
        )
    )
])

elastic_pipeline.fit(
    X_train,
    y_train
)

elastic_pred = elastic_pipeline.predict(
    X_test
)

NameError: name 'Pipeline' is not defined

In [8]:
elastic_mae = mean_absolute_error(
    y_test,
    elastic_pred
)

elastic_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        elastic_pred
    )
)

elastic_r2 = r2_score(
    y_test,
    elastic_pred
)

print("Elastic Net Regression")
print("MAE :", round(elastic_mae, 3))
print("RMSE:", round(elastic_rmse, 3))
print("R²  :", round(elastic_r2, 3))

NameError: name 'mean_absolute_error' is not defined

In [ ]:
elastic_model = elastic_pipeline.named_steps["model"]

elastic_coefficients = pd.DataFrame({
    "Feature": features,
    "Coefficient": elastic_model.coef_
})

elastic_coefficients.round(4)

# Effect of Alpha

We will test different alpha values.

Small alpha:
- Weak regularization

Large alpha:
- Strong regularization
- More coefficient shrinkage
- May produce more zero coefficients

In [ ]:
alphas = [
    0.001,
    0.01,
    0.1,
    1,
    10
]

alpha_results = []

for alpha in alphas:

    model = Pipeline([
        ("scaler", StandardScaler()),
        (
            "elastic",
            ElasticNet(
                alpha=alpha,
                l1_ratio=0.5,
                max_iter=10000
            )
        )
    ])

    model.fit(
        X_train,
        y_train
    )

    predictions = model.predict(
        X_test
    )

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    coefficients = (
        model.named_steps["elastic"].coef_
    )

    zero_count = np.sum(
        coefficients == 0
    )

    alpha_results.append({
        "Alpha": alpha,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2,
        "Zero Coefficients": zero_count
    })

alpha_results = pd.DataFrame(alpha_results)

alpha_results.round(4)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    alpha_results["Alpha"],
    alpha_results["R²"],
    marker="o"
)

plt.xscale("log")

plt.xlabel("Alpha")
plt.ylabel("R² Score")

plt.title(
    "Effect of Alpha on Elastic Net R²"
)

plt.grid(alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    alpha_results["Alpha"],
    alpha_results["Zero Coefficients"],
    marker="o"
)

plt.xscale("log")

plt.xlabel("Alpha")
plt.ylabel("Number of Zero Coefficients")

plt.title(
    "Effect of Alpha on Feature Selection"
)

plt.grid(alpha=0.3)

plt.show()

# Effect of L1 Ratio

The `l1_ratio` controls the balance between L1 and L2 regularization.

```text
l1_ratio = 0
→ Ridge-like

l1_ratio = 1
→ Lasso-like

l1_ratio = 0.5
→ Balanced L1 + L2

In [ ]:
l1_ratios = [
    0,
    0.25,
    0.5,
    0.75,
    1
]

ratio_results = []

for ratio in l1_ratios:

    model = Pipeline([
        ("scaler", StandardScaler()),
        (
            "elastic",
            ElasticNet(
                alpha=0.1,
                l1_ratio=ratio,
                max_iter=10000
            )
        )
    ])

    model.fit(
        X_train,
        y_train
    )

    predictions = model.predict(
        X_test
    )

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    coefficients = (
        model.named_steps["elastic"].coef_
    )

    zero_count = np.sum(
        coefficients == 0
    )

    ratio_results.append({
        "L1 Ratio": ratio,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2,
        "Zero Coefficients": zero_count
    })

ratio_results = pd.DataFrame(ratio_results)

ratio_results.round(4)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    ratio_results["L1 Ratio"],
    ratio_results["R²"],
    marker="o"
)

plt.xlabel("L1 Ratio")
plt.ylabel("R² Score")

plt.title(
    "Effect of L1 Ratio on Elastic Net R²"
)

plt.grid(alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    ratio_results["L1 Ratio"],
    ratio_results["Zero Coefficients"],
    marker="o"
)

plt.xlabel("L1 Ratio")
plt.ylabel("Number of Zero Coefficients")

plt.title(
    "Effect of L1 Ratio on Feature Selection"
)

plt.grid(alpha=0.3)

plt.show()

# Regularization Comparison

We will compare:

- Ridge
- Lasso
- Elastic Net

The goal is to understand how different regularization methods affect model performance and coefficients.

In [ ]:
from sklearn.linear_model import Ridge, Lasso

models = {
    "Ridge": Ridge(alpha=0.1),
    "Lasso": Lasso(alpha=0.1),
    "Elastic Net": ElasticNet(
        alpha=0.1,
        l1_ratio=0.5,
        max_iter=10000
    )
}

comparison_results = []

for name, estimator in models.items():

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", estimator)
    ])

    pipeline.fit(
        X_train,
        y_train
    )

    predictions = pipeline.predict(
        X_test
    )

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    comparison_results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    })

comparison = pd.DataFrame(
    comparison_results
)

comparison.round(4)

# Summary

## Elastic Net

Elastic Net combines:

- L1 Regularization
- L2 Regularization

## Alpha

Controls overall regularization strength.

```text
alpha ↑
→ Stronger regularization